# Semana 10: Mini-batches, optimizadores y PyTorch

**Pregunta de trabajo.** Reconocer el mismo loop en NumPy y PyTorch y comparar bajo un presupuesto declarado.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [2]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    print('PyTorch no disponible:', exc)

In [3]:
def one_hot(y, num_classes):
    y = np.asarray(y, dtype=int)
    out = np.zeros((len(y), num_classes), dtype=float)
    out[np.arange(len(y)), y] = 1.0
    return out


def relu(Z):
    return np.maximum(0.0, Z)


def relu_derivative(Z):
    return (Z > 0).astype(float)


def softmax(Z):
    Z = np.asarray(Z, dtype=float)
    shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def cross_entropy_loss(P, Y, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    return float(-np.sum(Y * np.log(P)) / len(Y))


def init_parameters(n_features, n_hidden, n_classes, seed=42):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0.0, np.sqrt(2.0 / n_features), size=(n_features, n_hidden))
    b1 = np.zeros(n_hidden)
    W2 = rng.normal(0.0, np.sqrt(2.0 / n_hidden), size=(n_hidden, n_classes))
    b2 = np.zeros(n_classes)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward_pass(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    P = softmax(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "P": P}
    return P, cache


def backward_pass(X, Y, cache, params, l2=0.0):
    n = len(X)
    P = cache["P"]
    A1 = cache["A1"]
    Z1 = cache["Z1"]
    W2 = params["W2"]

    dZ2 = (P - Y) / n
    dW2 = A1.T @ dZ2 + l2 * params["W2"]
    db2 = np.sum(dZ2, axis=0)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = X.T @ dZ1 + l2 * params["W1"]
    db1 = np.sum(dZ1, axis=0)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def update_parameters(params, grads, lr):
    return {key: value - lr * grads[key] for key, value in params.items()}


def predict(X, params):
    P, _ = forward_pass(X, params)
    return np.argmax(P, axis=1).astype(int)


def train_mlp(X_train, y_train, X_val, y_val, n_hidden=32, lr=0.1, epochs=150, l2=0.0, seed=42):
    n_classes = int(np.max(y_train)) + 1
    Y_train = one_hot(y_train, n_classes)
    params = init_parameters(X_train.shape[1], n_hidden, n_classes, seed=seed)
    history = []
    for epoch in range(epochs):
        P, cache = forward_pass(X_train, params)
        loss = cross_entropy_loss(P, Y_train)
        if l2:
            loss += 0.5 * l2 * (np.sum(params["W1"] ** 2) + np.sum(params["W2"] ** 2))
        grads = backward_pass(X_train, Y_train, cache, params, l2=l2)
        params = update_parameters(params, grads, lr=lr)
        if epoch % 10 == 0 or epoch == epochs - 1:
            train_acc = accuracy_score(y_train, predict(X_train, params))
            val_acc = accuracy_score(y_val, predict(X_val, params))
            history.append({"epoch": epoch, "loss": loss, "train_acc": train_acc, "val_acc": val_acc})
    return params, pd.DataFrame(history)

In [4]:
digits = load_digits()
X = digits.data.astype('float32') / 16.0
y = digits.target.astype('int64')
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

## Referencia NumPy

In [5]:
params, hist_np = train_mlp(X_train[:700], y_train[:700], X_val[:250], y_val[:250], epochs=100, lr=0.15, seed=5)
hist_np.tail(1)

,epoch,loss,train_acc,val_acc
10,99,0.433141,0.932857,0.856


## Loop PyTorch opcional

In [6]:
if TORCH_AVAILABLE:
    torch.manual_seed(5)
    model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    ds = TensorDataset(torch.tensor(X_train[:700]), torch.tensor(y_train[:700]))
    loader = DataLoader(ds, batch_size=64, shuffle=True)
    for epoch in range(10):
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        pred = model(torch.tensor(X_val[:250])).argmax(dim=1).numpy()
    print('val_acc_torch=', accuracy_score(y_val[:250], pred))
else:
    print('Use la comparación conceptual si PyTorch no está instalado.')

val_acc_torch= 0.944


## Estado de optimizadores y contraste real

Se verifica momentum con una secuencia conocida y se registra el entorno de PyTorch usado en clase.

In [7]:
def momentum_step(theta, gradient, velocity, alpha, beta=0.9):
    velocity = beta * velocity + (1 - beta) * gradient
    return theta - alpha * velocity, velocity

theta_m = np.array([0.0]); velocity = np.array([0.0])
states = []
for gradient in [np.array([4.0]), np.array([2.0])]:
    theta_m, velocity = momentum_step(theta_m, gradient, velocity, alpha=0.1)
    states.append((float(theta_m[0]), float(velocity[0])))
states

[(-0.039999999999999994, 0.3999999999999999),
 (-0.09599999999999997, 0.5599999999999998)]

In [8]:
comparison = {
    "torch_available": TORCH_AVAILABLE,
    "numpy_dev_accuracy": float(hist_np.iloc[-1]["val_acc"]),
}
if TORCH_AVAILABLE:
    comparison.update({
        "torch_version": torch.__version__,
        "torch_dev_accuracy": float(accuracy_score(y_val[:250], pred)),
        "loss_receives": "logits",
    })
comparison

{'torch_available': True,
 'numpy_dev_accuracy': 0.856,
 'torch_version': '2.2.2',
 'torch_dev_accuracy': 0.944,
 'loss_receives': 'logits'}